# ViKAG-Faith: LLM-judge và RAGAS với Qwen2.5-32B (Kaggle, 2 × T4)

1. Bên phải: **Settings → Accelerator → GPU T4 x2**, và **Internet → On** (cần tài khoản đã xác minh số điện thoại).
2. Bấm **Save Version → Save & Run All (Commit)**. Notebook chạy nền, đóng trình duyệt cũng được; tối đa 12 giờ.
3. Chạy xong, mở phiên bản đó → **Output** → tải `judges_32b.jsonl`, chép vào `vikag-faith/outputs/`, nhắn Claude.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%bash
set -eo pipefail
cd /kaggle/working
echo "1. kiểm tra Internet"; curl -sI -m 10 https://github.com > /dev/null || { echo "KHÔNG CÓ INTERNET: Settings -> Internet -> On (cần xác minh số điện thoại)"; exit 1; }
echo "2. tải mã nguồn"; [ -d vikag-faith ] || git clone -q --depth 1 https://github.com/vinhha195-oss/vikag-faith.git
echo "3. cài thư viện"; pip install -q ollama
echo "4. cài Ollama"; (apt-get -qq update && apt-get -qq install -y zstd) > /dev/null 2>&1 || true
command -v ollama > /dev/null || curl -fsSL https://ollama.com/install.sh | sh 2>&1 | tail -n 3
echo "5. chạy Ollama"; (nohup ollama serve > /kaggle/working/ollama.log 2>&1 &); sleep 10
echo "6. tải Qwen2.5-32B (khoảng 20 GB)"; ollama pull qwen2.5:32b 2>&1 | tail -n 2
ollama list

In [ ]:
%%bash
cd /kaggle/working/vikag-faith/src
export VIKAG_CPU=0 VIKAG_JUDGE_FILE=judges_32b.jsonl PYTHONIOENCODING=utf8
python s4_judges.py qwen2.5:32b 2>&1 | tail -n 5
cp ../outputs/judges_32b.jsonl /kaggle/working/judges_32b.jsonl
wc -l /kaggle/working/judges_32b.jsonl
nvidia-smi --query-gpu=memory.used --format=csv